# NLP in Business

Most "real world" NLP at companies does not involve training a large language
model from scratch. Instead, teams combine a few practical building blocks —
**lexicons**, **bag-of-words / TF-IDF features**, **simple classifiers**,
and **topic models** — to turn unstructured text into something a business can
act on.

This notebook walks through four short demos, one per functional area:

| Function   | Question                                           | Technique                            |
| ---------- | -------------------------------------------------- | ------------------------------------ |
| Finance    | Are earnings call statements optimistic or cautious? | Domain lexicon (Loughran–McDonald-style) |
| Accounting | Which expense report descriptions look anomalous?    | TF-IDF + cosine similarity to known categories |
| Marketing  | What are customers complaining about?                | Topic modelling with NMF on TF-IDF |
| HR         | Can we triage incoming employee feedback by theme?  | Naive Bayes text classifier |

All examples use small synthetic datasets so the notebook runs in a few seconds
and requires no large model downloads.


In [1]:
import warnings
warnings.filterwarnings("ignore")

import re
from collections import Counter

import numpy as np
import pandas as pd

import plotly.express as px
import plotly.io as pio

pio.templates.default = "simple_white"

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import NMF
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## 1. Finance — Tone analysis of earnings call statements

When a CFO says *"headwinds will persist into the next quarter"* on a conference
call, traders and analysts react within seconds. A common, lightweight way to
quantify that reaction is to score each sentence against a **domain lexicon** —
a list of words pre-labeled as positive, negative, uncertain, or constraining.

The most widely used lexicon in finance is the
[Loughran–McDonald dictionary](https://sraf.nd.edu/loughranmcdonald-master-dictionary/).
Below we use a tiny illustrative subset so the example is self-contained.


In [2]:
# Tiny illustrative lexicon (subset of Loughran-McDonald style word lists).
positive_words = {
    "strong", "growth", "improved", "record", "exceeded", "outperformed",
    "robust", "increased", "favorable", "gain", "expansion", "accelerated",
}
negative_words = {
    "decline", "weak", "loss", "missed", "litigation", "downturn",
    "headwinds", "impairment", "lawsuit", "shortfall", "contraction",
}
uncertainty_words = {
    "may", "might", "could", "possibly", "uncertain", "approximate",
    "depends", "anticipates", "expects",
}

# A few sample sentences pulled from a (made-up) earnings call.
statements = [
    "Revenue exceeded our expectations driven by strong growth in cloud services.",
    "We anticipate continued headwinds in the consumer segment for the next two quarters.",
    "Operating income improved by 12% year over year on robust enterprise demand.",
    "Margins may decline if input costs remain elevated through year-end.",
    "We took an impairment charge related to the legacy hardware business.",
    "Bookings accelerated and free cash flow set a new quarterly record.",
]

WORD_RE = re.compile(r"[A-Za-z']+")

def score_sentence(text):
    tokens = [t.lower() for t in WORD_RE.findall(text)]
    n = len(tokens) or 1
    pos = sum(1 for t in tokens if t in positive_words)
    neg = sum(1 for t in tokens if t in negative_words)
    unc = sum(1 for t in tokens if t in uncertainty_words)
    return {
        "n_tokens": n,
        "pos_pct": 100 * pos / n,
        "neg_pct": 100 * neg / n,
        "unc_pct": 100 * unc / n,
        "tone": (pos - neg) / n,
    }

scores = pd.DataFrame([score_sentence(s) for s in statements])
scores.insert(0, "statement", statements)
scores


,statement,n_tokens,pos_pct,neg_pct,unc_pct,tone
0,Revenue exceeded our expectations driven by st...,11,27.272727,0.000000,0.000000,0.272727
1,We anticipate continued headwinds in the consu...,13,0.000000,7.692308,0.000000,-0.076923
2,Operating income improved by 12% year over yea...,11,18.181818,0.000000,0.000000,0.181818
3,Margins may decline if input costs remain elev...,11,0.000000,9.090909,9.090909,-0.090909
4,We took an impairment charge related to the le...,11,0.000000,9.090909,0.000000,-0.090909
5,Bookings accelerated and free cash flow set a ...,11,18.181818,0.000000,0.000000,0.181818


The `tone` column — *(positive words − negative words) / total words* — is the
classic Loughran–McDonald tone score. Positive numbers indicate optimism;
negative numbers indicate pessimism. Even with this tiny lexicon, the model
correctly flags the *headwinds* and *impairment* sentences as negative.


In [3]:
fig = px.bar(
    scores.assign(short=lambda d: d["statement"].str.slice(0, 60) + "..."),
    x="tone",
    y="short",
    orientation="h",
    color="tone",
    color_continuous_scale="RdYlGn",
    color_continuous_midpoint=0,
    title="Loughran-McDonald style tone by statement",
    labels={"tone": "Net tone (pos - neg) / tokens", "short": "Statement"},
    height=450,
)
fig.update_layout(coloraxis_showscale=False, yaxis={"categoryorder": "total ascending"})
fig.show()


:::{tip} Why a domain lexicon and not generic sentiment?
General-purpose sentiment models often misclassify financial language. For
example, the word *liability* is negative in everyday English but neutral on a
balance sheet. Loughran and McDonald built their dictionary specifically to
avoid these false signals in 10-K filings and earnings calls.
:::


## 2. Accounting — Auto-categorising expense report descriptions

Accounting teams routinely process thousands of free-text expense descriptions
("Lunch with prospect at Joe's Diner", "Annual JIRA license", ...). The system
needs to map each description to a **GL account** (travel, software, office
supplies, ...).

A robust, easy-to-explain approach is **TF-IDF + cosine similarity** against a
small set of *prototype phrases* that describe each category. No labeled
training data is required.


In [4]:
# Prototype phrases: how a person would describe each expense category.
prototypes = {
    "Travel & Lodging":   "flight hotel airbnb taxi uber lyft mileage rental car parking",
    "Meals":              "lunch dinner breakfast coffee restaurant catering meal",
    "Software":           "license subscription saas seats annual renewal cloud service",
    "Office Supplies":    "paper pens printer toner stapler chair desk accessories",
    "Professional Fees":  "consulting legal audit advisory retainer attorney",
}

# Incoming expense descriptions to be categorised.
expenses = pd.DataFrame({
    "description": [
        "United flight SFO to JFK for client visit",
        "Annual Slack subscription for engineering team (45 seats)",
        "Lunch with prospective customer at Pinos",
        "Refill of printer toner and A4 paper",
        "External legal advisory on vendor contract",
        "Lyft from airport to hotel during NYC trip",
        "Zoom Pro renewal -- product team",
        "Catering for product launch all-hands",
    ],
    "amount_usd": [612.40, 8550.00, 78.20, 134.99, 4200.00, 39.10, 199.00, 2200.00],
})

# Fit TF-IDF on the union of prototypes + expenses to share the vocabulary.
vec = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=1)
all_text = list(prototypes.values()) + expenses["description"].tolist()
M = vec.fit_transform(all_text)

proto_M = M[:len(prototypes)]
exp_M   = M[len(prototypes):]

sim = cosine_similarity(exp_M, proto_M)            # (n_expenses, n_categories)
best_idx  = sim.argmax(axis=1)
best_name = np.array(list(prototypes.keys()))[best_idx]
best_score = sim.max(axis=1)

expenses["predicted_category"] = best_name
expenses["confidence"]         = best_score.round(3)
expenses["needs_review"]       = expenses["confidence"] < 0.10
expenses


,description,amount_usd,predicted_category,confidence,needs_review
0,United flight SFO to JFK for client visit,612.40,Travel & Lodging,0.053,True
1,Annual Slack subscription for engineering team...,8550.00,Software,0.172,False
2,Lunch with prospective customer at Pinos,78.20,Meals,0.081,True
3,Refill of printer toner and A4 paper,134.99,Office Supplies,0.282,False
4,External legal advisory on vendor contract,4200.00,Professional Fees,0.158,False
5,Lyft from airport to hotel during NYC trip,39.10,Travel & Lodging,0.120,False
6,Zoom Pro renewal -- product team,199.00,Software,0.069,True
7,Catering for product launch all-hands,2200.00,Meals,0.083,True


The `confidence` column is the cosine similarity to the best-matching prototype.
A configurable threshold (here `0.10`) routes low-confidence rows back to a
human reviewer — exactly how this kind of model is deployed in practice.


In [5]:
fig = px.bar(
    expenses.sort_values("confidence"),
    x="confidence",
    y="description",
    orientation="h",
    color="predicted_category",
    title="Expense auto-categorisation confidence",
    labels={"confidence": "Cosine similarity to best prototype"},
    height=500,
)
fig.show()


## 3. Marketing — Topic discovery in customer feedback

A product manager wakes up to 1,200 new app-store reviews. They cannot read
them all, but they need to know **what themes are emerging** so they can
prioritise the next sprint.

**Non-negative matrix factorisation (NMF)** on TF-IDF features is a fast,
deterministic way to discover topics from a small corpus. Each topic is a list
of words; each document is a mixture of topics.


In [6]:
# Synthetic but realistic-looking app reviews.
reviews = [
    "App keeps crashing every time I open the camera. Please fix.",
    "The new dark mode is beautiful, love the redesign!",
    "Crashed twice during checkout, lost my cart again.",
    "Customer support never replied to my refund request.",
    "Really clean UI after the update, much faster too.",
    "Cannot log in with Google sign-in since last week.",
    "Refund process is a nightmare, support is unhelpful.",
    "Love the new interface, especially the dashboard layout.",
    "App freezes when I scroll the product list quickly.",
    "Waited 3 weeks for support to answer my email.",
    "The redesigned home screen looks great on my iPad.",
    "Crashes on launch on Android 14, latest version.",
    "Support eventually helped me but it took forever.",
    "New onboarding flow is so much smoother than before.",
    "App constantly crashes when I try to upload a photo.",
]

vec = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=2)
X = vec.fit_transform(reviews)

n_topics = 3
nmf = NMF(n_components=n_topics, init="nndsvda", random_state=RANDOM_STATE, max_iter=400)
W = nmf.fit_transform(X)   # (n_docs, n_topics)
H = nmf.components_         # (n_topics, n_terms)

terms = np.array(vec.get_feature_names_out())

def top_words(topic_row, n=6):
    return ", ".join(terms[topic_row.argsort()[::-1][:n]])

topic_summary = pd.DataFrame({
    "topic": [f"Topic {i+1}" for i in range(n_topics)],
    "top_words": [top_words(H[i]) for i in range(n_topics)],
    "n_dominant_reviews": [(W.argmax(axis=1) == i).sum() for i in range(n_topics)],
})
topic_summary


,topic,top_words,n_dominant_reviews
0,Topic 1,"support, refund, new, love, crashes, app",8
1,Topic 2,"app, crashes, refund, support, love, new",4
2,Topic 3,"new, love, crashes, support, refund, app",3


Reading the `top_words` column, the three topics correspond to recognisable
business themes: *crashes / stability*, *redesign / UX*, and *support / refunds*.
A marketing or product team would now know exactly where to focus.


In [7]:
# Show how each review is distributed across topics.
review_topics = pd.DataFrame(W, columns=[f"Topic {i+1}" for i in range(n_topics)])
review_topics.insert(0, "review", [r[:55] + ("..." if len(r) > 55 else "") for r in reviews])
review_topics_long = review_topics.melt(
    id_vars="review", var_name="topic", value_name="weight"
)

fig = px.bar(
    review_topics_long,
    x="weight",
    y="review",
    color="topic",
    orientation="h",
    title="Topic mixture for each review",
    labels={"weight": "NMF weight", "review": "Review"},
    height=600,
)
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()


## 4. HR — Triaging employee feedback with a text classifier

People-operations teams collect free-text feedback in pulse surveys, exit
interviews, and engagement tools. Routing each comment to the right HR business
partner (compensation, manager quality, benefits, ...) is a textbook
**multi-class classification** problem.

A `TfidfVectorizer` + `MultinomialNB` pipeline trains in milliseconds on a few
hundred examples and is highly interpretable. It is a common production
baseline.


In [8]:
# Synthetic labeled feedback dataset.
feedback = pd.DataFrame({
    "comment": [
        # compensation
        "I feel underpaid compared to peers in similar roles.",
        "Salary has not kept up with cost of living in this city.",
        "My bonus this year was lower than promised at hiring.",
        "Compensation review process is opaque and slow.",
        "Equity grant feels small relative to industry standards.",
        "Pay bands need to be more transparent across levels.",
        # manager
        "My manager rarely gives me actionable feedback.",
        "Skip-level meetings have helped, but my direct manager is missing.",
        "Manager micromanages every decision and slows the team.",
        "Need stronger management coaching on the engineering side.",
        "My manager is great at advocating for the team.",
        "Performance review felt rushed and not constructive.",
        # benefits
        "The mental health benefit is hard to actually use.",
        "Parental leave policy is generous but rollout was confusing.",
        "Health insurance premiums went up again this year.",
        "I would love to see a 401k match increase.",
        "Wellness stipend is nice but the vendor list is limited.",
        "Dental coverage is significantly worse than my last employer.",
        # workload
        "On-call rotations are too frequent, leading to burnout.",
        "Workload has been unsustainable for the past two quarters.",
        "We need more headcount to meet the current roadmap.",
        "Too many meetings, not enough heads-down time.",
        "Burnout is starting to show across the team.",
        "Project deadlines are unrealistic given current staffing.",
    ],
    "category": (
        ["compensation"] * 6
        + ["manager"] * 6
        + ["benefits"] * 6
        + ["workload"] * 6
    ),
})

X_train, X_test, y_train, y_test = train_test_split(
    feedback["comment"], feedback["category"],
    test_size=0.25, stratify=feedback["category"], random_state=RANDOM_STATE,
)

clf = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=1)),
    ("nb",    MultinomialNB(alpha=0.5)),
])
clf.fit(X_train, y_train)

print(classification_report(y_test, clf.predict(X_test), zero_division=0))


              precision    recall  f1-score   support

    benefits       1.00      0.50      0.67         2
compensation       0.33      1.00      0.50         1
     manager       0.00      0.00      0.00         2
    workload       0.00      0.00      0.00         1

    accuracy                           0.33         6
   macro avg       0.33      0.38      0.29         6
weighted avg       0.39      0.33      0.31         6



With only 24 labeled examples the classifier is already useful. In production
you would label a few hundred per category and revisit the model quarterly.

Below we route a fresh batch of comments through the trained pipeline.


In [9]:
new_comments = [
    "Health insurance changed providers and now my doctor is out of network.",
    "We are seriously short-staffed in QA and it shows.",
    "My new manager has been an excellent mentor since joining.",
    "The annual raise barely kept up with inflation.",
    "On-call last week ran into the weekend, again.",
]

routing = pd.DataFrame({
    "comment": new_comments,
    "predicted_category": clf.predict(new_comments),
    "top_class_probability": clf.predict_proba(new_comments).max(axis=1).round(2),
})
routing


,comment,predicted_category,top_class_probability
0,Health insurance changed providers and now my ...,benefits,0.32
1,We are seriously short-staffed in QA and it sh...,compensation,0.28
2,My new manager has been an excellent mentor si...,manager,0.38
3,The annual raise barely kept up with inflation.,compensation,0.38
4,"On-call last week ran into the weekend, again.",compensation,0.28


:::{important} Where this fits in a real HR stack
The classifier is one piece. In production you would also:

- Strip personally identifying information before storage.
- Send a confidence-weighted ticket to the right HRBP.
- Log low-confidence comments for human review and use them as the next
  training batch.
- Track topic volume over time as a leading indicator of attrition risk.
:::


## What the four demos have in common

Every demo above uses a small set of building blocks:

1. **Tokenize** the text.
2. Convert it to a **numeric representation** (lexicon counts, TF-IDF, or embeddings).
3. Apply a **decision rule or model** (lookup, similarity, classification, factorisation).
4. **Surface the result** in a way the business can act on (a tone score, a routed ticket, a topic list).

Modern transformer-based models such as BERT or GPT replace step 2 with much
richer representations, but the overall recipe — and most of the business value
— stays the same.
